# Stochastic Calculus from Zero · Episode 26: The whole curve: Hull-White and HJM

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **Today's curve** (Part 4): Series 1's illustrative AONIA curve, valued 25 September 2026 (`quotes_illustrative.csv`), with Hull-White mean reversion 3% and volatility 0.94% (Series 4's calibration to its illustrative AUD swaptions, rounded). Shaped like a plausible market, made up for teaching; not market data.
>
> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Before we start · 3. Today's curve · 4. Hull-White: a drift that fits · 5. It reprices the curve · 6. HJM: model the whole curve · 7. The drift is not a choice · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 26,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative AONIA curve (made up for teaching)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))


# ---- Part 4 additions -------------------------------------------------------------------------------------------
#   Hull-White fitted to today's curve P0(t), in the G1++ form r(t) = x(t) + phi(t), dx = -a x dt + sigma dW, x(0) = 0.
def hw_B(t, T, a):
    return (1 - np.exp(-a * (T - t))) / a


def hw_V(t, T, a, sigma):
    """Variance of the integral of x from t to T, given x(t)."""
    tau = T - t
    return sigma**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 1.5 / a)


def hw_bond(t, T, x, P0, a, sigma):
    """Zero-coupon bond P(t, T) given the state x(t); P0 is today's discount curve."""
    return P0(T) / P0(t) * np.exp(0.5 * (hw_V(t, T, a, sigma) - hw_V(0, T, a, sigma) + hw_V(0, t, a, sigma))
                                  - hw_B(t, T, a) * np.asarray(x))


def hw_phi(t, f0, a, sigma):
    """The deterministic shift: r(t) = x(t) + phi(t) reprices today's curve (f0 = today's instantaneous forwards)."""
    return f0(t) + sigma**2 / (2 * a**2) * (1 - math.exp(-a * t))**2


def hw_exact_step(x, I, dt, a, sigma, z):
    """One exact step of (x, integral of x) over dt; z has shape (n, 2) of independent standard normals."""
    e = math.exp(-a * dt)
    bd = (1 - e) / a
    vx = sigma**2 / (2 * a) * (1 - e * e)
    vi = sigma**2 / a**2 * (dt - 2 * bd + (1 - e * e) / (2 * a))
    cxi = sigma**2 / (2 * a**2) * (1 - e)**2
    eps = z @ np.linalg.cholesky([[vx, cxi], [cxi, vi]]).T
    return x * e + eps[:, 0], I + x * bd + eps[:, 1]


#   Jumps: Merton's model, log jump sizes normal with mean mu_j and standard deviation d_j, lam jumps a year.
def merton_call(s, k, t, r, vol, lam, mu_j, d_j, terms=60):
    """Merton call price: Black-Scholes prices weighted by the Poisson chance of n jumps."""
    kbar = math.exp(mu_j + 0.5 * d_j**2) - 1
    lp = lam * (1 + kbar) * t
    total, w = 0.0, math.exp(-lp)
    for n in range(terms):
        if n:
            w *= lp / n
        total = total + w * bs_call(s, k, t, r - lam * kbar + n * math.log(1 + kbar) / t,
                                    math.sqrt(vol**2 + n * d_j**2 / t))[0]
    return total


def bs_implied_vol(price, s, k, t, r, lo=0.01, hi=1.5):
    """Black-Scholes implied volatility by bisection."""
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float(bs_call(s, k, t, r, mid)[0]) < price else (lo, mid)
    return 0.5 * (lo + hi)

Series 1's AONIA curve (`episodes/_shared/aud_curves.py`). The curve code expects `today`, `cal` and `dc`:

In [ ]:
today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()
out["meta"]["valuation_date"] = valuation_date

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

In [ ]:
curves, handles, _, _ = build_curves(pd.read_csv(aud_quotes_file))
aonia, aonia_h = curves["AONIA"], handles["AONIA"]
P0 = lambda t: aonia.discount(float(t)) if t > 0 else 1.0                                     # today's discount curve
f0 = lambda t: aonia.forwardRate(float(t), float(t), ql.Continuous, ql.NoFrequency).rate()    # instantaneous forwards
A_HW, SIG_HW = 0.03, 0.0094                                                                   # Hull-White, Series 4 (rounded)

## 2. Before we start

The Part 3 checkpoint's answers are in the Episode 25 notebook and at the back of the Part 3 book. This episode uses Part 3's Vasicek bond formula (Episode 23) and the idea that discounted prices are martingales (Episodes 21 and 25).

## 3. Today's curve

Series 1's AONIA curve rises from 3.85% overnight to 4.50% at 10 years. Part 3's Vasicek model, with its constant long-run level, produces one particular shape of curve, and it isn't this one. Below: continuously compounded zero rates $-\ln P(0, T)/T$ from both.

In [ ]:
mats = np.round(np.arange(0.25, 10.01, 0.25), 2)
zero_aonia = np.array([-math.log(P0(T)) / T for T in mats])
zero_vas = np.array([-math.log(float(vasicek_bond(T, 0.0385, 0.30, 0.035, 0.01))) / T for T in mats])
out["curve"] = {"T": mats.tolist(), "aonia_pct": (zero_aonia * 100).round(4).tolist(), "vasicek_pct": (zero_vas * 100).round(4).tolist(),
                "z1_pct": float(zero_aonia[3] * 100), "z10_pct": float(zero_aonia[-1] * 100),
                "v10_pct": float(zero_vas[-1] * 100), "f0_pct": f0(0.0) * 100,
                "ticks": [{"x": x, "label": f"{x}Y"} for x in (1, 2, 3, 5, 7, 10)]}
{k: v for k, v in out["curve"].items() if not isinstance(v, list)}

## 4. Hull-White: a drift that fits

Hull and White (1990) let the target level move with time:

$$dr = \big(\theta(t) - a\,r\big)\,dt + \sigma\,dW .$$

Choosing

$$\theta(t) = \frac{\partial f(0, t)}{\partial t} + a\,f(0, t) + \frac{\sigma^2}{2a}\big(1 - e^{-2at}\big),$$

where $f(0, t)$ is today's instantaneous forward curve, makes the model reprice every zero-coupon bond on today's curve. $\theta(t)/a$ plays the role of Vasicek's long-run level, now one for each date. In the notebook we write $r = x + \varphi(t)$ with $dx = -ax\,dt + \sigma\,dW$, $x(0) = 0$ (`hw_bond`, `hw_phi`), which is the same model.

In [ ]:
def theta(t, h=1e-4):
    lo = max(t - h, 0.0)
    return (f0(t + h) - f0(lo)) / (t + h - lo) + A_HW * f0(t) + SIG_HW**2 / (2 * A_HW) * (1 - math.exp(-2 * A_HW * t))

tt = np.round(np.arange(0.25, 10.01, 0.25), 2)
th = np.array([theta(t) for t in tt])
out["theta"] = {"t": tt.tolist(), "level_pct": (th / A_HW * 100).round(4).tolist(),
                "f0_pct": [round(f0(t) * 100, 4) for t in tt],
                "rows": [{"t": t, "level_pct": theta(t) / A_HW * 100, "f0_pct": f0(t) * 100} for t in (0.5, 1, 2, 5, 10)],
                "a_pct": A_HW * 100, "sigma_pct": SIG_HW * 100}
out["years"] = {"hull_white": 1990, "hjm": 1992}
pd.DataFrame(out["theta"]["rows"])

## 5. It reprices the curve

QuantLib's `HullWhite` model takes today's curve and does the fitting itself. Its zero-coupon bonds against the curve's own discount factors:

In [ ]:
hw = ql.HullWhite(aonia_h, A_HW, SIG_HW)
zeros = [{"T": T, "curve": P0(T), "hw": hw.discountBond(0.0, float(T), f0(0.0)),
          "formula": float(hw_bond(0.0, T, 0.0, P0, A_HW, SIG_HW))} for T in (1, 2, 3, 5, 7, 10)]
assert all(abs(z["hw"] - z["curve"]) < 1e-10 and abs(z["formula"] - z["curve"]) < 1e-12 for z in zeros)
out["zeros"] = {"rows": zeros, "max_gap": max(abs(z["hw"] - z["curve"]) for z in zeros)}
pd.DataFrame(zeros)

## 6. HJM: model the whole curve

Heath, Jarrow and Morton (1992) model every forward rate at once:

$$df(t, T) = \alpha(t, T)\,dt + \sigma(t, T)\,dW_t \qquad \text{for every maturity } T > t,$$

with the short rate $r(t) = f(t, t)$ and bond prices $P(t, T) = \exp\big(-\int_t^T f(t, u)\,du\big)$. Today's curve is the starting point, so it fits by construction.

**The HJM drift condition.** Under the risk-neutral measure, $P(t, T)/B(t)$ must be a martingale for every $T$. Applying Itô to $-\int_t^T f(t, u)\,du$ and setting the drift of $P/B$ to zero gives

$$\alpha(t, T) = \sigma(t, T)\int_t^T \sigma(t, u)\,du .$$

You choose the volatility; no-arbitrage chooses the drift. With $\sigma(t, T) = \sigma e^{-a(T - t)}$ this is exactly Hull-White.

## 7. The drift is not a choice

Simulate the whole forward curve, weekly, for 2 years, on a weekly grid of maturities out to 10 years, with $\sigma(t, T) = 0.94\% \times e^{-0.03(T - t)}$. Then check the martingale property: $E[D(0, 2)\,P(2, 10)]$ should equal today's $P(0, 10)$. Run it twice on the **same** random draws: once with the HJM drift, once with no drift at all.

In [ ]:
rng = np.random.default_rng(seed)
h, T_max, t_end, n_paths = 1 / 52, 10.0, 2.0, 20_000
grid = np.arange(0, T_max + 1e-9, h)
f_today = np.array([f0(T) for T in grid])
target = math.exp(-(f_today[:-1] * h).sum())             # P(0, 10) on the same grid
steps = int(round(t_end / h))
dW = rng.standard_normal((steps, n_paths)) * math.sqrt(h)

def simulate(with_drift, keep=0):
    f = np.tile(f_today, (n_paths, 1))
    logD = np.zeros(n_paths)
    for k in range(steps):
        tau = grid[k:] - k * h
        sig = SIG_HW * np.exp(-A_HW * tau)
        alpha = sig * SIG_HW * (1 - np.exp(-A_HW * tau)) / A_HW if with_drift else 0.0 * sig
        logD -= f[:, k] * h                               # the short rate is f(t, t)
        f[:, k:] += alpha * h + np.outer(dW[k], sig)
    P_t = np.exp(-(f[:, steps:-1] * h).sum(axis=1))
    return np.exp(logD) * P_t, f[:keep, steps:]

v_hjm, f_keep = simulate(True, keep=30)
v_zero, _ = simulate(False)
diff = v_zero - v_hjm
hjm = {"paths": n_paths, "target": target, "curve": P0(T_max),
       "hjm": float(v_hjm.mean()), "hjm_se": float(v_hjm.std() / math.sqrt(n_paths)),
       "zero": float(v_zero.mean()), "zero_se": float(v_zero.std() / math.sqrt(n_paths)),
       "hjm_bp": (v_hjm.mean() / target - 1) * 1e4, "zero_bp": (v_zero.mean() / target - 1) * 1e4,
       "diff": float(diff.mean()), "diff_se": float(diff.std() / math.sqrt(n_paths)),
       "T_fan": grid[steps::13].round(3).tolist(), "fan": (f_keep[:, ::13] * 100).round(3).tolist(),
       "today_fan": (f_today[steps::13] * 100).round(3).tolist(),
       "ticks": [{"x": x, "label": f"{x}Y"} for x in (2, 4, 6, 8, 10)]}
assert abs(hjm["hjm"] - target) < 3 * hjm["hjm_se"]
assert hjm["diff"] > 10 * hjm["diff_se"]
out["hjm"] = hjm
{k: v for k, v in hjm.items() if not isinstance(v, list)}

## 8. This episode's question

If $\sigma(t, T) = \sigma$ (no decay), the drift is $\alpha = \sigma \int_t^T \sigma\,du = \sigma^2(T - t)$: the **Ho-Lee** model.

In [ ]:
out["question"] = {"model": "Ho-Lee", "drift": "sigma^2 (T - t)"}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Derive the HJM drift condition. Write $Y_t = -\int_t^T f(t, u)\,du$, so $P = e^Y$; find $dY$, then require the drift of $P/B$ to vanish.

<details><summary>Answer</summary>

$dY = f(t, t)\,dt - \big(\int_t^T\alpha\,du\big)dt - \big(\int_t^T\sigma\,du\big)dW$. By Itô, $dP/P = dY + \tfrac12(dY)^2$, so the drift of $P/P$ is $r - \int_t^T\alpha\,du + \tfrac12\big(\int_t^T\sigma\,du\big)^2$. For $P/B$ to be a martingale this must equal $r$: $\int_t^T\alpha\,du = \tfrac12\big(\int_t^T\sigma\,du\big)^2$. Differentiate in $T$: $\alpha(t, T) = \sigma(t, T)\int_t^T\sigma(t, u)\,du$.

</details>

**Exercise 2 (By hand).** Show that $\sigma(t, T) = \sigma e^{-a(T - t)}$ gives the HJM drift $\frac{\sigma^2}{a}e^{-a(T - t)}\big(1 - e^{-a(T - t)}\big)$.

<details><summary>Answer</summary>

$\int_t^T\sigma e^{-a(u - t)}\,du = \frac{\sigma}{a}\big(1 - e^{-a(T - t)}\big)$; multiply by $\sigma e^{-a(T - t)}$.

</details>

**Exercise 3 (By hand).** Why can't Part 3's Vasicek model (constant $\theta$) fit an arbitrary curve?

<details><summary>Answer</summary>

Its bond prices $A(\tau)e^{-B(\tau)r_0}$ depend on four numbers ($r_0, a, \theta, \sigma$), so it can only produce a four-parameter family of curves. A curve built from twelve market quotes generally isn't in that family; a time-dependent $\theta(t)$ gives one free number per date.

</details>

**Exercise 4 (In code).** For a flat 4% curve, compute $\theta(t)/a$ at 1, 5 and 10 years. Where does the extra over 4% come from?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
lv = [0.04 + SIG_HW**2 / (2 * A_HW**2) * (1 - math.exp(-2 * A_HW * t)) for t in (1, 5, 10)]
assert all(l > 0.04 for l in lv)
print([f"{l:.4%}" for l in lv], "- the convexity term sigma^2/(2a^2)(1 - e^(-2at)) lifts the level above the forward")

**Exercise 5 (In code).** Rerun the HJM check with half the HJM drift. Where does the answer land?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
f = np.tile(f_today, (n_paths, 1)); logD = np.zeros(n_paths)
for k in range(steps):
    tau = grid[k:] - k * h
    sig = SIG_HW * np.exp(-A_HW * tau)
    logD -= f[:, k] * h
    f[:, k:] += 0.5 * sig * SIG_HW * (1 - np.exp(-A_HW * tau)) / A_HW * h + np.outer(dW[k], sig)
half = float((np.exp(logD) * np.exp(-(f[:, steps:-1] * h).sum(axis=1))).mean())
assert hjm["hjm"] < half < hjm["zero"]
print(f"full drift {hjm['hjm']:.5f}, half {half:.5f}, none {hjm['zero']:.5f}; today's P(0,10) {target:.5f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")